In [7]:
import numpy as np
import pandas as pd
import librosa
import os

In [8]:
rows = []

for language in os.listdir('./Dataset'):
    lang_path = os.path.join('./Dataset', language)
    if not os.path.isdir(lang_path):
        continue

    for gender in ['Male', 'Female']:
        gender_path = os.path.join(lang_path, gender)
        if not os.path.isdir(gender_path):
            continue

        for file in os.listdir(gender_path):
            if file.lower().endswith('.mp3'):
                rows.append({
                    'filepath': os.path.join(gender_path, file),
                    'target': language
                })

file_paths = pd.DataFrame(rows)
file_paths

,filepath,target
0,./Dataset\German\Male\810100147_male_german_vo...,German
1,./Dataset\German\Male\810100147_male_german_vo...,German
2,./Dataset\German\Male\810100147_male_german_vo...,German
3,./Dataset\German\Male\810100147_male_german_vo...,German
4,./Dataset\German\Male\810100147_male_german_vo...,German
...,...,...
715,./Dataset\Spanish\Female\810800006_female_span...,Spanish
716,./Dataset\Spanish\Female\810800006_female_span...,Spanish
717,./Dataset\Spanish\Female\810800006_female_span...,Spanish
718,./Dataset\Spanish\Female\810800006_female_span...,Spanish


In [9]:
def extract_features(file_path):
    audio, sr = librosa.load(file_path, sr=22050)

    # top_db=20 means anything below 20db relative to max is considered silence
    audio, _ = librosa.effects.trim(audio, top_db=20)
    
    # MFCC (Mel-frequency cepstral coefficients) - The shape of the sound
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    mfccs_mean = np.mean(mfccs.T, axis=0)
    mfccs_var = np.var(mfccs.T, axis=0)
    
    # Spectral Centroid - The center of mass of the sound (Brightness)
    cent = librosa.feature.spectral_centroid(y=audio, sr=sr)
    cent_mean = np.mean(cent)
    cent_var = np.var(cent)
    
    # Zero Crossing Rate - Noisiness/Percussiveness
    zcr = librosa.feature.zero_crossing_rate(audio)
    zcr_mean = np.mean(zcr)
    zcr_var = np.var(zcr)
    
    # RMS Energy - Loudness
    rms = librosa.feature.rms(y=audio)
    rms_mean = np.mean(rms)
    rms_var = np.var(rms)

    feature_vector = np.hstack([mfccs_mean, mfccs_var, cent_mean, cent_var, zcr_mean, zcr_var, rms_mean, rms_var])
    
    return feature_vector

In [10]:
def create_dataset(df_path):
    data = []

    for file_path, language in zip(df_path['filepath'], df_path['target']):
        features = extract_features(file_path)
        _ = np.hstack((features, language))
        data.append(_)
        
    return data

In [11]:
df = pd.DataFrame(create_dataset(file_paths))
df

c:\Users\kamya\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


,0,1,2,3,4,5,6,7,8,9,...,37,38,39,40,41,42,43,44,45,46
0,-307.2024841308594,73.51306915283203,-4.735387802124023,34.61341094970703,4.925637722015381,8.435842514038086,-6.880800247192383,5.771332263946533,4.517052173614502,7.7932233810424805,...,81.63368225097656,94.10591125488281,67.13753509521484,2249.933150680687,1355771.4954518955,0.11626887628976784,0.008820786239378734,0.05239337682723999,0.0022115656174719334,German
1,-320.6261901855469,79.6156005859375,2.2733237743377686,35.96018600463867,7.918567657470703,10.991131782531738,-6.31997537612915,4.049417018890381,2.1558094024658203,6.214526176452637,...,68.18925476074219,117.31729888916016,67.33624267578125,2085.200221131979,1456163.6911706622,0.1046432811887975,0.008824004553786551,0.043192971497774124,0.0014225523918867111,German
2,-352.5018310546875,93.53467559814453,3.3081555366516113,32.70573043823242,5.722683906555176,11.876784324645996,-0.9357349276542664,4.1038007736206055,4.41855001449585,6.298893451690674,...,72.04994201660156,126.02293395996094,61.47459030151367,1837.594008517297,1220549.4457973042,0.08738563637411942,0.006518046475216272,0.03721030056476593,0.0011854765471071005,German
3,-333.33428955078125,89.4705581665039,2.6262903213500977,38.64476013183594,9.224559783935547,10.798149108886719,-4.4715576171875,3.378542900085449,1.402066707611084,6.156917095184326,...,61.870670318603516,106.85172271728516,60.838294982910156,1896.781393353466,1186896.8012971072,0.08786125197270701,0.0058501984304390655,0.041715074330568314,0.0012438142439350486,German
4,-347.8489685058594,87.04412841796875,4.772493839263916,40.264190673828125,11.757944107055664,14.127995491027832,-4.642753601074219,4.775572299957275,1.0375287532806396,6.997015476226807,...,66.51852416992188,103.20206451416016,64.5245590209961,1934.4721908434258,1547134.6552945073,0.09488540540381722,0.00954353344055969,0.04033101722598076,0.0013284204760566354,German
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,-334.2942810058594,97.18785858154297,1.8970063924789429,25.751840591430664,3.8313610553741455,6.24647855758667,-6.05432653427124,-0.086042121052742,-7.294749736785889,-1.9754974842071533,...,89.99580383300781,63.107791900634766,68.6166763305664,1632.9988810541852,1565955.831561802,0.06979606414744646,0.010431362723156526,0.06033739820122719,0.0021142063196748495,Spanish
716,-337.90655517578125,90.0734634399414,2.144426107406616,26.452816009521484,4.420064926147461,3.709589958190918,-5.430403232574463,0.18960082530975342,-7.5571675300598145,-2.9939372539520264,...,84.58812713623047,62.56423568725586,68.31320190429688,1623.0176531361872,1695743.5673502074,0.07103635363449579,0.011350266730807995,0.05988854914903641,0.0022235633805394173,Spanish
717,-351.1990051269531,92.22593688964844,0.8302161693572998,25.06365966796875,4.728697776794434,3.4342479705810547,-5.006286144256592,0.4982372224330902,-5.687548637390137,-2.5245423316955566,...,94.00184631347656,73.34522247314453,96.70372009277344,1556.7702010108985,1222101.1574874858,0.05933549168634396,0.00804999852586588,0.05919378995895386,0.0025473206769675016,Spanish
718,-328.4798278808594,90.0936050415039,0.6355485320091248,25.208904266357422,4.162844657897949,5.089888095855713,-8.39030647277832,-0.6643893122673035,-7.316596508026123,-3.9086551666259766,...,99.18807983398438,84.222412109375,85.8350601196289,1711.8305883847179,1596866.2228012152,0.07217148645364306,0.010305648272925115,0.06492392718791962,0.002574072452262044,Spanish


In [12]:
df.to_csv('Dataset.csv')